# Fase 4b — Treinamento do Ensemble

Treina `N_MODELS` instâncias independentes do `HotelReviewClassifier` e salva cada checkpoint em `models/saved/ensemble/`. Na inferência, `EnsembleInference` carrega todos e agrega as predições por média de probabilidades.

## Fluxo do notebook

| Célula | O que faz |
|---|---|
| 1. Setup | Seed base, device, paths, número de modelos |
| 2. Dataset e DataLoaders | Idêntico ao `03_training.ipynb` |
| 3. Treinamento do ensemble | Chama `train_ensemble` — treina N modelos com seeds distintas |
| 4. Resultado | Exibe F1 de cada modelo e confirma os checkpoints salvos |
| 5. Teste de inferência | Carrega o ensemble e faz uma predição de exemplo |

> Custo: `N_MODELS × tempo de um treino`. Com `N_MODELS=3` e GPU RTX 3050 (~54 min/epoch × 5 epochs), esperar ~13h no total.

## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import os
import torch

N_MODELS      = 3
BASE_SEED     = 0       # seeds usadas: 0, 1, 2, ..., N_MODELS-1
EPOCHS        = 5
LR            = 2e-5
BATCH_SIZE    = 32
VAL_RATIO     = 0.2
CSV_PATH      = '../data/processed/reviews_labeled.csv'
ENSEMBLE_DIR  = '../models/saved/ensemble'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Modelos a treinar: {N_MODELS}  |  Seeds: {list(range(BASE_SEED, BASE_SEED + N_MODELS))}')

## 2. Dataset e DataLoaders

Mesmo setup do `03_training.ipynb`. O split treino/val é fixado pela `BASE_SEED` — cada modelo do ensemble vê a mesma divisão de dados, mas com inicialização de pesos diferente.

In [ ]:
from torch.utils.data import DataLoader, random_split
from src.data.dataset import ReviewDataset

dataset = ReviewDataset(CSV_PATH)

val_size   = int(len(dataset) * VAL_RATIO)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(BASE_SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Treino:    {train_size:,} reviews  ({len(train_loader)} batches)')
print(f'Validacao: {val_size:,} reviews  ({len(val_loader)} batches)')

## 3. Treinamento do Ensemble

`train_ensemble` itera de `i=0` até `N_MODELS-1`:
- Cria um `HotelReviewClassifier` novo (pesos aleatórios)
- Chama `train_single_model` com `seed = BASE_SEED + i`
- Salva o melhor checkpoint (por F1 de validação) em `ENSEMBLE_DIR/model_{i}.pt`

O resultado é uma lista com o F1 de cada modelo.

In [ ]:
from src.models.classifier import HotelReviewClassifier
from src.models.trainer import train_ensemble

results = train_ensemble(
    model_factory=HotelReviewClassifier,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    save_dir=ENSEMBLE_DIR,
    n_models=N_MODELS,
    base_seed=BASE_SEED,
    n_epochs=EPOCHS,
    lr=LR,
)

## 4. Resultado

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nF1 medio do ensemble: {df['f1_macro'].mean():.4f} (+/- {df['f1_macro'].std():.4f})")

print('\nCheckpoints salvos:')
for r in results:
    size_mb = os.path.getsize(r['path']) / 1024 / 1024
    print(f"  {r['path']}  ({size_mb:.1f} MB)")

## 5. Teste de Inferência

Carrega o ensemble e roda uma predição de exemplo para confirmar que os checkpoints foram salvos corretamente e que a agregação funciona.

In [ ]:
from src.models.ensemble import EnsembleInference

checkpoint_paths = [r['path'] for r in results]
ensemble = EnsembleInference(checkpoint_paths, device=str(DEVICE))

test_reviews = [
    'The room was very clean and the staff was incredibly helpful.',
    'Terrible experience. The bathroom was dirty and no hot water.',
    'Location was great but the wifi did not work at all.',
]

for review in test_reviews:
    pred = ensemble.predict(review)
    print(f'Review : {review[:60]}...')
    print(f'  Sentimento : {pred["sentiment"]}')
    print(f'  Rating     : {pred["rating"]}')
    print(f'  Prioridade : {pred["priority"]}')
    print(f'  Categorias : {pred["categories"]}')
    print()